# Path-Orphan — end-to-end Colab runner (verified, complete)
Detect **conditional / orphan essentials** in *R. solanacearum* GMI1000 + build a fused function atlas. Latest sandbox results already in repo:
- **Bar:** conservation rogue R@P30 = **0.000**
- **dN/dS:** real-data **NEGATIVE** (ω 0.158 vs 0.160)
- **ESM:** within-org 0.843 → leave-one-clade-out **0.000** (audited: PCA artifact + homology leak)
- **2×2 lens (novelty #1):** all 3 non-circular predictions PASS
- **Fused atlas (novelty #2):** live-ghost kill-gate PASS; 355 prioritized hits
- **Co-inheritance:** 49 confound-guarded function hints; validator GATE PASS (20% recovery vs 1.8% null, p<0.001)
- **DEG cross-grade:** overall recall 84%; rogue recall 34% (independent screen)

**Remaining (this notebook):** structure (Foldseek), GEM (mechanistic extrapolator), specific-phenotype miner.

## Setup (CPU runtime)

In [ ]:
import subprocess, os
from pathlib import Path
REPO=Path('/content/cell'); BRANCH='claude/vectorize-gex-propensity-NRqBW'
if not REPO.exists():
    subprocess.run(['git','clone','-b',BRANCH,'https://github.com/Nikku03/cell.git',str(REPO)],check=True)
else:
    subprocess.run(['git','-C',str(REPO),'pull','origin',BRANCH],check=True)
os.chdir(REPO)
from google.colab import drive; drive.mount('/content/drive')
DRIVE=Path('/content/drive/MyDrive/path_orphan'); DRIVE.mkdir(parents=True,exist_ok=True)
FEBA_SRC=Path('/content/drive/MyDrive/cell_count_dynamics/multiorg/fitness_browser/feba.db')
if FEBA_SRC.exists() and not Path('/content/feba.db').exists():
    subprocess.run(['cp',str(FEBA_SRC),'/content/feba.db'],check=True)
print('feba.db present:', Path('/content/feba.db').exists())
subprocess.run(['pip','-q','install','pandas','pyarrow','scikit-learn','xgboost','requests','cobra'],check=True)
def stash(glob_):
    import shutil
    for p in Path('outputs/orphan').glob(glob_):
        if p.is_file(): shutil.copy(p, DRIVE/p.name)

## Verify (re-run all 15 smoke tests)

In [ ]:
for s in ['bridge','baseline','dnds','foldseek','cofit','model','esm','loo',
         '2x2','atlas','afdb','gem','coinherit','deg_grade','validator','phenotypes']:
    print('==', s); subprocess.run(['python',f'scripts/orphan_{s}.py','--smoke'],check=True)

## Phase A — cached steps (instant from Drive)

In [ ]:
!python scripts/orphan_bridge.py    --real
!python scripts/orphan_baseline.py  --real
!python scripts/orphan_dnds.py      --real
!python scripts/orphan_cofit.py     --real --feba /content/feba.db
!python scripts/orphan_coinherit.py --real
stash('bridge_*'); stash('baseline_*'); stash('dnds_*'); stash('proteins_*')
stash('uniprot_request_*'); stash('cofit_*'); stash('coinherit_*')

## Phase A — STEP 1 structure channel (~2 hr first time, ~25 GB AFDB DB to Drive once)

In [ ]:
# UniProt idmapping RefSeq_Protein -> UniProtKB
import requests, time, pathlib
ids=pathlib.Path('outputs/orphan/uniprot_request_beril_RalstoniaGMI1000.txt').read_text().split()
r=requests.post('https://rest.uniprot.org/idmapping/run',
    data={'from':'RefSeq_Protein','to':'UniProtKB','ids':','.join(ids)})
jid=r.json()['jobId']; print('jobId',jid)
while True:
    s=requests.get(f'https://rest.uniprot.org/idmapping/status/{jid}').json()
    if 'results' in s or s.get('jobStatus')=='FINISHED': break
    time.sleep(5)
res=requests.get(f'https://rest.uniprot.org/idmapping/results/{jid}?size=500&format=tsv').text
pathlib.Path('outputs/orphan/uniprot_map.tsv').write_text(res)
print('mapped',len(res.splitlines())-1,'ids')
# install foldseek
!wget -q https://mmseqs.com/foldseek/foldseek-linux-avx2.tar.gz && tar xzf foldseek-linux-avx2.tar.gz
os.environ['PATH']=f"{os.getcwd()}/foldseek/bin:{os.environ['PATH']}"
# AFDB pull (parallel, resume-safe)
!mkdir -p queries
!awk 'NR>1{print $2}' outputs/orphan/uniprot_map.tsv | sort -u | \
  xargs -P 8 -I{} sh -c 'test -f queries/AF-{}-F1-model_v4.pdb || wget -q -O queries/AF-{}-F1-model_v4.pdb https://alphafold.ebi.ac.uk/files/AF-{}-F1-model_v4.pdb'
# AFDB-cluster DB (cache to Drive)
if not pathlib.Path('/content/drive/MyDrive/path_orphan/afdb_clust').exists():
    !foldseek databases Alphafold/UniProt50 afdb_clust tmp
    !cp -r afdb_clust* /content/drive/MyDrive/path_orphan/
else:
    !cp -r /content/drive/MyDrive/path_orphan/afdb_clust* ./
!foldseek easy-search queries/ afdb_clust outputs/orphan/foldseek_result.m8 tmp \
    --format-output 'query,target,fident,evalue,bits,alntmscore'
# per-query pLDDT
import sys; sys.path.insert(0,'scripts')
from orphan_afdb import mean_plddt
with open('outputs/orphan/plddt.tsv','w') as f:
    for p in pathlib.Path('queries').glob('AF-*-F1-model_v4.pdb'):
        f.write(f'{p.stem.split("-")[1]}\t{mean_plddt(p.read_text()):.1f}\n')
!python scripts/orphan_foldseek.py --real --m8 outputs/orphan/foldseek_result.m8 \
    --plddt outputs/orphan/plddt.tsv
stash('foldhit_*'); stash('foldseek_result.m8'); stash('plddt.tsv')

## Phase A — STEP 6 specific-phenotype miner (Price 2018-style)

In [ ]:
!python scripts/orphan_phenotypes.py --real --feba /content/feba.db
stash('phenotypes_*')

## Phase A — STEP 2 GEM mechanistic rung (~1 day; gapseq build is the slow part)

In [ ]:
# gapseq build (slowest step ~hours)
!conda install -y -c bioconda -c conda-forge gapseq 2>/dev/null || pip install gapseq
!gapseq doall outputs/orphan/proteins_beril_RalstoniaGMI1000.faa
# single-gene-deletion per FB condition
import cobra, pandas as pd, sqlite3, sys; sys.path.insert(0,'scripts')
from orphan_gem import condition_to_media
from cobra.flux_analysis import single_gene_deletion
model=cobra.io.read_sbml_model('proteins_beril_RalstoniaGMI1000.xml')
con=sqlite3.connect('/content/feba.db')
conds=pd.read_sql("SELECT DISTINCT condition_1, media, aerobic FROM Experiment WHERE orgId='RalstoniaGMI1000'",con)
con.close()
rows=[]
for _,c in conds.iterrows():
    open_ex=condition_to_media(c['condition_1'], c['media'])
    with model:
        for r in model.exchanges: r.lower_bound=0
        for ex in open_ex:
            if ex in model.reactions: model.reactions.get_by_id(ex).lower_bound=-10
        d=single_gene_deletion(model)
        for g,row in d.iterrows():
            rows.append((g,c['condition_1'],row['growth']<0.01))
pd.DataFrame(rows,columns=['locus_tag','condition','in_silico_essential']).to_parquet(
    'outputs/orphan/gem_essentiality_beril_RalstoniaGMI1000.parquet')
stash('gem_essentiality_*')

## ⚠️ SWITCH RUNTIME → GPU for ESM (CACHED — only re-run if --force)

In [ ]:
!pip -q install torch transformers
# already on Drive as esm_all.parquet (full 1280-d, 32392 proteins); skip unless rebuilding
import shutil, pathlib
if pathlib.Path('/content/drive/MyDrive/path_orphan/esm_all.parquet').exists():
    shutil.copy('/content/drive/MyDrive/path_orphan/esm_all.parquet','outputs/orphan/esm_all.parquet')
    print('ESM cached on Drive; skip')
else:
    !python scripts/orphan_esm.py --real --multi --batch 32
    stash('esm_all.parquet')

## ⚠️ SWITCH RUNTIME → CPU — Phase C: fused atlas + validator + DEG grade + writeup

In [ ]:
# restore all features from Drive
import shutil; from pathlib import Path
for p in Path('/content/drive/MyDrive/path_orphan').iterdir():
    if p.is_file() and p.suffix in ('.parquet','.csv','.json','.faa','.tsv','.m8'):
        shutil.copy(p,'outputs/orphan/'+p.name)
# build the fused atlas with ALL channels available
!python scripts/orphan_atlas.py --real
stash('atlas_*'); stash('live_ghost_hitlist_*')
# function-recovery kill-gate (the trust bound)
!python scripts/orphan_validator.py --real --n_sample 400 --n_perms 200
stash('validator_*')
# DEG cross-grade (independent screen)
!python scripts/orphan_deg_grade.py --real
stash('deg_grade_*')
# 2x2 lens (instant, on bridge+dnds)
!python scripts/orphan_2x2.py --real
stash('twobytwo_*')
# LOO + perm null (the cross-clade negative)
!python scripts/orphan_loo.py --real --l2 5
!python scripts/orphan_loo.py --real --l2 5 --clade_regex Ralstonia
stash('loo_*')
import json
for f in ['atlas_beril_RalstoniaGMI1000_summary','validator_beril_RalstoniaGMI1000','deg_grade_beril_RalstoniaGMI1000','twobytwo_beril_RalstoniaGMI1000']:
    print('===',f); print(json.dumps(json.load(open(f'outputs/orphan/{f}.json')),indent=2))

## Read the result
- `atlas_*_summary.json` → function-tier distribution (annotated / fold_named / phenotype_module / context_inferred / live_ghost / unknown)
- `validator_*.json` → recovery vs shuffled-pair null; **gate must PASS** for the atlas to be trusted
- `deg_grade_*.json` → independent-screen precision/recall
- `twobytwo_*.json` → the acute × evolutionary lens, all 3 non-circular predictions
- `loo_*_beril_RalstoniaGMI1000.json` → the leakage-controlled cross-clade negative (the paper-1 capstone)
- `live_ghost_hitlist_*.csv` → prioritized experimental targets